# 07 - Build Direct Lake gold aggregates

Rebuilds one observation-date, one video capture-date, and one processing-date partition. Backfill orchestration passes historical dates explicitly; steady-state orchestration passes dates touched by newly committed work. Observation date is derived from each crossing timestamp so videos spanning UTC midnight aggregate correctly.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
FLOW_DATE = ""  # YYYY-MM-DD observation-time partition to rebuild
CAPTURE_DATE = ""  # YYYY-MM-DD video capture partition to rebuild
OPERATION_DATE = ""  # YYYY-MM-DD completion/queue date to rebuild
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import date, datetime, timezone
import json
import re

import notebookutils
from pyspark.sql import DataFrame, SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parse_date(value: str, name: str) -> date:
    try:
        return date.fromisoformat(value)
    except ValueError as error:
        raise ValueError(f"{name} must be YYYY-MM-DD") from error


flow_date = parse_date(FLOW_DATE, "FLOW_DATE")
capture_date = parse_date(CAPTURE_DATE, "CAPTURE_DATE")
operation_date = parse_date(OPERATION_DATE, "OPERATION_DATE")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)


def replace_partition(target_table: str, source: DataFrame, date_column: str, value: date) -> int:
    rows = source.count()
    (
        source.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"{date_column} = DATE '{value.isoformat()}'")
        .saveAsTable(target_table)
    )
    return rows


lines = spark_session.table(table("line_counts_committed")).where(
    F.to_date("observed_at_utc") == F.lit(flow_date)
)
flow_minute = (
    lines.withColumn("minute_utc", F.date_trunc("minute", "observed_at_utc"))
    .groupBy("minute_utc", "camera_id", "location_id")
    .agg(
        F.sum("frame_in_count").cast("long").alias("entries"),
        F.sum("frame_out_count").cast("long").alias("exits"),
        F.countDistinct("work_id").cast("long").alias("source_videos"),
    )
    .withColumn("net_flow", F.col("entries") - F.col("exits"))
    .withColumn("refreshed_at", F.lit(now))
    .withColumn("flow_date", F.lit(flow_date))
    .select(
        "minute_utc", "camera_id", "location_id", "entries", "exits",
        "net_flow", "source_videos", "refreshed_at", "flow_date",
    )
)
flow_hour = (
    lines.withColumn("hour_utc", F.date_trunc("hour", "observed_at_utc"))
    .groupBy("hour_utc", "camera_id", "location_id")
    .agg(
        F.sum("frame_in_count").cast("long").alias("entries"),
        F.sum("frame_out_count").cast("long").alias("exits"),
        F.countDistinct("work_id").cast("long").alias("source_videos"),
    )
    .withColumn("net_flow", F.col("entries") - F.col("exits"))
    .withColumn("refreshed_at", F.lit(now))
    .withColumn("flow_date", F.lit(flow_date))
)
runs = spark_session.table(table("runs_committed")).where(
    F.to_date("captured_at_utc") == F.lit(capture_date)
)
gold_video = runs.select(
    "work_id",
    "captured_at_utc",
    "camera_id",
    "location_id",
    F.col("duration_seconds").alias("video_duration_seconds"),
    "processing_seconds",
    F.when(
        (F.col("duration_seconds") > 0) & (F.col("processing_seconds") > 0),
        F.col("duration_seconds") / F.col("processing_seconds"),
    ).alias("speed_x_realtime"),
    "distinct_people",
    "line_in_count",
    "line_out_count",
    "completed_at",
    F.lit(capture_date).alias("capture_date"),
)

attempts = spark_session.table(table("video_attempts"))
work = spark_session.table(table("video_work"))
committed = work.where(F.col("status") == "SUCCEEDED").select(
    "work_id",
    F.col("committed_attempt_id").alias("committed_attempt_id"),
)
attempts_with_commit = attempts.alias("a").join(committed.alias("w"), "work_id", "left")
claimed = attempts.where(F.to_date("claimed_at") == F.lit(operation_date)).groupBy(
    F.date_trunc("hour", "claimed_at").alias("hour_utc")
).agg(F.count("attempt_id").cast("long").alias("started"))
completed = attempts_with_commit.where(F.to_date("completed_at") == F.lit(operation_date)).groupBy(
    F.date_trunc("hour", "completed_at").alias("hour_utc")
).agg(
    F.sum(
        F.when(
            (F.col("status") == "SUCCEEDED") & (F.col("attempt_id") == F.col("committed_attempt_id")),
            1,
        ).otherwise(0)
    ).cast("long").alias("succeeded"),
    F.sum(F.when(F.col("status").isin("RETRY_WAIT", "TERMINAL_FAILED", "DEAD_LETTERED"), 1).otherwise(0)).cast("long").alias("failed"),
    (
        F.sum(
            F.when(
                (F.col("status") == "SUCCEEDED") & (F.col("attempt_id") == F.col("committed_attempt_id")),
                F.col("source_duration_seconds"),
            ).otherwise(0.0)
        ) / 3600.0
    ).alias("video_hours_completed"),
    F.avg("processing_seconds").alias("average_processing_seconds"),
    F.expr("percentile_approx(processing_seconds, 0.95)").alias("p95_processing_seconds"),
)
queued = work.where(F.to_date("queued_at") == F.lit(operation_date)).groupBy(
    F.date_trunc("hour", "queued_at").alias("hour_utc")
).agg(F.count("work_id").cast("long").alias("queued"))
operations = (
    queued.join(claimed, "hour_utc", "full")
    .join(completed, "hour_utc", "full")
    .fillna({"queued": 0, "started": 0, "succeeded": 0, "failed": 0, "video_hours_completed": 0.0})
    .withColumn("refreshed_at", F.lit(now))
    .withColumn("operation_date", F.lit(operation_date))
    .select(
        "hour_utc", "queued", "started", "succeeded", "failed",
        "video_hours_completed", "average_processing_seconds",
        "p95_processing_seconds", "refreshed_at", "operation_date",
    )
)

outcome = {
    "flow_date": flow_date.isoformat(),
    "capture_date": capture_date.isoformat(),
    "operation_date": operation_date.isoformat(),
    "flow_minute_rows": replace_partition(table("gold_flow_minute"), flow_minute, "flow_date", flow_date),
    "flow_hour_rows": replace_partition(table("gold_flow_hour"), flow_hour, "flow_date", flow_date),
    "video_rows": replace_partition(table("gold_video"), gold_video, "capture_date", capture_date),
    "operations_rows": replace_partition(table("gold_operations_hour"), operations, "operation_date", operation_date),
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))